# Exploração do modelo de credibilidade

Carrega a calibração, simula os cenários para alguns valores de λ e mostra as trajetórias. Rode a partir da raiz do projeto (`modelo-credibilidade/`).

In [ ]:
import os, sys
if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')
sys.path.insert(0, '.')
import numpy as np, pandas as pd, yaml, matplotlib.pyplot as plt
from src import modelo, varredura, metricas
p = modelo.carregar_calibracao('config/calibracao.yaml', especificacao='P1')  # P1, P2 ou P3
cfg = yaml.safe_load(open('config/cenarios.yaml', encoding='utf-8'))
cens = varredura.preparar_cenarios(p, cfg)
{k: round(p[k], 2) for k in ['i_ef0', 's_estab0', 'prem0', 'tp0', 'r_long_neutro']}

## Painel histórico trimestral

In [ ]:
painel = pd.read_csv('data/processed/trimestral.csv', index_col=0, parse_dates=True)
painel[['meta_selic', 'ipca12', 'pi_e', 'ntnb10', 'dbgg']].loc['2010':].plot(subplots=True, figsize=(9, 9));

## Trajetórias determinísticas por cenário e λ

In [ ]:
lams = np.array([0.2, 0.5, 0.8, 1.0])
fig, axs = plt.subplots(3, 3, figsize=(12, 9), sharex=True)
for j, (k, c) in enumerate(cens.items()):
    tr = modelo.simular(p, c['cen'], lams, plano=c['plano'])  # λ_fiscal = λ_π (diagonal)
    for i, v in enumerate(['d_gross', 'pi', 'i']):
        for n, l in enumerate(lams):
            axs[i, j].plot(tr[v][n], label=f'λ={l}')
        axs[i, j].set_title(f"{c['cen']['nome']}: {v}", fontsize=9)
axs[0, 0].legend(); plt.tight_layout()

## Critério de sucesso na grade determinística

In [ ]:
det = varredura.diagonal(p, cens, varredura.grade_lambda(0.05), None)
det.pivot(index='lambda', columns='cenario', values='p_sucesso')

## Superfície λ_π × λ_fiscal (um cenário)

In [ ]:
g = varredura.grade_lambda(0.1)
lp, lf = np.meshgrid(g, g, indexing='ij')
c = cens['C']
tr = modelo.simular(p, c['cen'], lp.ravel(), lf.ravel(), plano=c['plano'], guardar=('d_gross', 'pi', 'R'))
ok, _ = metricas.sucesso(tr, p)
pd.DataFrame(ok.reshape(lp.shape).astype(int), index=g, columns=g)  # linhas: λ_π; colunas: λ_fiscal